In [1]:
import numpy as np

tags = ["Determiner", "Adjective", "Noun", "Verb"]

words = ["The", "little", "boy", "eats", "an", "apple"]

word_to_index = {
    "The": 0,
    "little": 1,
    "boy": 2,
    "eats": 3,
    "an": 4,
    "apple": 5
}

init_prob = np.array([
    0.7,
    0.1,
    0.1,
    0.1
])

transition = np.array([
    [0.05, 0.60, 0.30, 0.05],
    [0.05, 0.10, 0.80, 0.05],
    [0.05, 0.05, 0.10, 0.80],
    [0.70, 0.05, 0.15, 0.10]
])

emission = np.array([
    [0.50, 0.00, 0.00, 0.00, 0.50, 0.00],
    [0.00, 1.00, 0.00, 0.00, 0.00, 0.00],
    [0.00, 0.00, 0.50, 0.00, 0.00, 0.50],
    [0.00, 0.00, 0.00, 1.00, 0.00, 0.00]
])

log_init = np.log(init_prob + 1e-10)
log_trans = np.log(transition + 1e-10)
log_emit = np.log(emission + 1e-10)

def viterbi(sentence):
    T = len(sentence)
    N = len(tags)

    dp = np.full((N, T), -np.inf)
    backpointer = np.zeros((N, T), dtype=int)

    dp[:, 0] = log_init + log_emit[:, sentence[0]]

    for t in range(1, T):
        for j in range(N):
            scores = dp[:, t-1] + log_trans[:, j] + log_emit[j, sentence[t]]
            dp[j, t] = np.max(scores)
            backpointer[j, t] = np.argmax(scores)

    best_path = [np.argmax(dp[:, -1])]

    for t in range(T - 1, 0, -1):
        best_path.append(backpointer[best_path[-1], t])

    best_path.reverse()
    return best_path

sentence = ["The", "little", "boy", "eats", "an", "apple"]

sentence_index = [word_to_index[word] for word in sentence]

predicted = viterbi(sentence_index)

print("Sentence:")
print(" ".join(sentence))

print("\nPredicted POS Tags:")
for word, tag in zip(sentence, predicted):
    print(f"{word:<8} -> {tags[tag]}")

true_tags = ["Determiner", "Adjective", "Noun", "Verb", "Determiner", "Noun"]

predicted_tags = [tags[i] for i in predicted]

correct = sum(p == t for p, t in zip(predicted_tags, true_tags))
accuracy = correct / len(true_tags)

print(f"\nAccuracy: {accuracy*100:.2f}%")

Sentence:
The little boy eats an apple

Predicted POS Tags:
The      -> Determiner
little   -> Adjective
boy      -> Noun
eats     -> Verb
an       -> Determiner
apple    -> Noun

Accuracy: 100.00%
